# ATC-ASR Benchmark

In [15]:
%pip install -q transformers>=4.35 datasets jiwer soundfile accelerate numpy silero-vad torchao>=0.16

In [16]:
import torch
import json
import time
import re
from pathlib import Path
from collections import Counter
import numpy as np
from datasets import load_dataset
from transformers import pipeline as hf_pipeline
from jiwer import wer, cer, process_words

In [28]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/atc_asr_output")
except Exception:
    OUTPUT_DIR = Path("atc_asr_output")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "configs").mkdir(exist_ok=True)
(OUTPUT_DIR / "predictions").mkdir(exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Normalizer

In [18]:
DIGIT_TO_WORD = {
    "0":"zero","1":"one","2":"two","3":"three","4":"four",
    "5":"five","6":"six","7":"seven","8":"eight","9":"nine",
}
WORD_TO_DIGIT = {v:k for k,v in DIGIT_TO_WORD.items()}
WORD_TO_DIGIT["niner"] = "9"


def normalize_for_wer(text: str) -> str:
    t = text.lower().strip()
    t = re.sub(r"[,\.!?;:\"\(\)]", " ", t)
    t = re.sub(r"\bniner\b", "nine", t)
    t = re.sub(r"\balfa\b", "alpha", t)
    t = re.sub(r"\btree\b", "three", t)
    t = re.sub(r"\bfife\b", "five", t)

    # some airlines
    for pattern, repl in [
        (r"\bryan\s+air\b",   "ryanair"),
        (r"\beuro\s+wings\b", "eurowings"),
        (r"\bsky\s+travel\b", "skytravel"),
        (r"\bspeed\s+bird\b", "speedbird"),
        (r"\bbel\s+avia\b",   "belavia"),
        (r"\bairfrans\b",     "airfrance"),
    ]:
        t = re.sub(pattern, repl, t)
    t = re.sub(r"\bok\b", "okay", t)

    def expand_fl(m):
        return "flight level " + " ".join(DIGIT_TO_WORD[d] for d in m.group(1))
    t = re.sub(r"\bfl\s*(\d{2,3})\b", expand_fl, t)
    def expand_rwy(m):
        side = {"l":"left","r":"right","c":"center"}.get((m.group(2) or "").lower(), "")
        words = " ".join(DIGIT_TO_WORD[d] for d in m.group(1))
        return ("runway " + words + (" " + side if side else "")).strip()
    t = re.sub(r"\brwy\s*(\d{1,2})([lrc]?)\b", expand_rwy, t)
    t = re.sub(r"\b(\d+)\b", lambda m: " ".join(DIGIT_TO_WORD[d] for d in m.group(1)), t)
    return re.sub(r"\s+", " ", t).strip()


def normalize_for_display(text: str) -> str:
    t = text.strip()
    num_pat = '|'.join(re.escape(w) for w in WORD_TO_DIGIT.keys())

    def compress_fl(m):
        digits = [WORD_TO_DIGIT[w] for w in m.group(1).split() if w in WORD_TO_DIGIT]
        return ('FL' + ''.join(digits)) if digits else m.group(0)
    t = re.sub(
        rf'\bflight level\s+((?:{num_pat})(?:\s+(?:{num_pat}))*)',
        compress_fl, t, flags=re.IGNORECASE
    )

    def compress_rwy(m):
        digits = [WORD_TO_DIGIT[w] for w in m.group(1).split() if w in WORD_TO_DIGIT]
        side = {'left':'L','right':'R','center':'C'}.get((m.group(2) or '').lower(), '')
        return ('RWY' + ''.join(digits) + side) if digits else m.group(0)
    t = re.sub(
        rf'\brunway\s+((?:{num_pat})(?:\s+(?:{num_pat}))*)\s*(left|right|center)?',
        compress_rwy, t, flags=re.IGNORECASE
    )

    def compress_freq(m):
        left  = ''.join(WORD_TO_DIGIT.get(w,'') for w in m.group(1).split())
        right = ''.join(WORD_TO_DIGIT.get(w,'') for w in m.group(2).split())
        return left + '.' + right
    t = re.sub(
        rf'((?:{num_pat})(?:\s+(?:{num_pat}))*)\s+decimal\s+((?:{num_pat})(?:\s+(?:{num_pat}))*)',
        compress_freq, t, flags=re.IGNORECASE
    )
    return t.strip()

## VAD

In [19]:
from silero_vad import load_silero_vad, get_speech_timestamps

_vad_model = None

def get_vad_model():
    global _vad_model
    if _vad_model is None:
        _vad_model = load_silero_vad()
    return _vad_model


def apply_vad(audio_array: np.ndarray, vad_cfg: dict) -> list[np.ndarray]:
    """
    اگر vad_cfg["enabled"] = False باشد، کل صدا را بدون تغییر برمی‌گرداند.
    اگر True باشد، بخش‌های گفتار را با Silero VAD جدا می‌کند.
    """
    if not vad_cfg.get("enabled", False):
        return [audio_array]

    model = get_vad_model()
    audio_tensor = torch.from_numpy(audio_array.copy()).float()

    timestamps = get_speech_timestamps(
        audio_tensor,
        model,
        sampling_rate=16000,
        threshold=vad_cfg.get("threshold", 0.5),
        min_speech_duration_ms=vad_cfg.get("min_speech_duration_ms", 250),
        min_silence_duration_ms=vad_cfg.get("min_silence_duration_ms", 100),
    )

    if not timestamps:
        return [audio_array]   # نویز خالص → کل صدا را برگردان

    return [audio_array[ts["start"]:ts["end"]] for ts in timestamps]


print("VAD utility آماده است")
print("  enabled=False → کل صدا بدون تغییر پردازش می‌شود")
print("  enabled=True  → سکوت‌ها حذف، گفتار تشخیص داده می‌شود")

VAD utility آماده است
  enabled=False → کل صدا بدون تغییر پردازش می‌شود
  enabled=True  → سکوت‌ها حذف، گفتار تشخیص داده می‌شود


## Dataset

In [20]:
DATASET_ID = "Jzuluaga/atco2_corpus_1h"

try:
    dataset = load_dataset(DATASET_ID, split="test", trust_remote_code=True)
except Exception:
    ds_all = load_dataset(DATASET_ID, trust_remote_code=True)
    dataset = ds_all[list(ds_all.keys())[0]]

TEXT_KEY = "text" if "text" in dataset[0] else "transcription"

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Jzuluaga/atco2_corpus_1h' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'Jzuluaga/atco2_corpus_1h' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


## Config

In [29]:

whisper_large_v3_jacktol = "jacktol/whisper-large-v3-finetuned-for-ATC"
whisper_large_v3_fjmg    = "fjmgAI/whisper-large-v3-ATC"
whisper_medium_en        = "jacktol/whisper-medium.en-fine-tuned-for-ATC"

RUNS = [
    {
        "run_id": "run1",
        "name": "jacktol-whisper-large-v3-ATC | beam=5",
        "model_id": whisper_large_v3_jacktol,
        "model_type": "whisper",
        "vad_config": {"enabled": False},
        "generate_kwargs": {
            "language": "english",
            "task": "transcribe",
            "temperature": 0.0,
            "num_beams": 5,
        },
    },
    {
        "run_id": "run2",
        "name": "jacktol-whisper-large-v3-ATC | beam=1 (greedy)",
        "model_id": whisper_large_v3_jacktol,
        "model_type": "whisper",
        "vad_config": {"enabled": False},
        "generate_kwargs": {
            "language": "english",
            "task": "transcribe",
            "temperature": 0.0,
            "num_beams": 1,
        },
    },
    {
        "run_id": "run3",
        "name": "jacktol-whisper-medium.en-ATC | beam=1 (greedy)",
        "model_id": whisper_medium_en,
        "model_type": "whisper",
        "vad_config": {"enabled": False},
        "generate_kwargs": {
            "temperature": 0.0,
            "num_beams": 1,
        },
    },
    {
        "run_id": "run4",
        "name": "fjmgAI-whisper-large-v3-ATC | beam=1",
        "model_id": whisper_large_v3_fjmg,
        "model_type": "whisper",
        "vad_config": {"enabled": False},
        "generate_kwargs": {
            "language": "english",
            "task": "transcribe",
            "temperature": 0.0,
            "num_beams": 1,
        },
    },
    # ── Non-Whisper ──
    {
        "run_id": "run5",
        "name": "XLS-R-300M | UWB-ATCC+ATCOSIM (CTC)",
        "model_id": "Jzuluaga/wav2vec2-xls-r-300m-en-atc-uwb-atcc-and-atcosim",
        "model_type": "ctc",
        "vad_config": {"enabled": False},
        "generate_kwargs": {},
    },
    # ── VAD demo run ──
    {
        "run_id": "run6",
        "name": "jacktol-whisper-large-v3-ATC | beam=5 + VAD",
        "model_id": whisper_large_v3_jacktol,
        "model_type": "whisper",
        "vad_config": {
            "enabled": True,
            "threshold": 0.5,
            "min_speech_duration_ms": 250,
            "min_silence_duration_ms": 100,
        },
        "generate_kwargs": {
            "language": "english",
            "task": "transcribe",
            "temperature": 0.0,
            "num_beams": 5,
        },
    },
]

for run in RUNS:
    cfg = {**run, "dataset": DATASET_ID, "normalization": "normalize_for_wer"}
    path = OUTPUT_DIR / "configs" / f"config_{run['run_id']}.json"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)



## Inference

In [30]:
import gc

all_records   = {}
_loaded_pipes = {}


def run_inference(run_cfg, dataset):
    model_id   = run_cfg["model_id"]
    model_type = run_cfg.get("model_type", "whisper")
    vad_cfg    = run_cfg.get("vad_config", {"enabled": False})

    if model_id not in _loaded_pipes:
        print(f"  Loading [{model_type}]: {model_id}")
        _loaded_pipes[model_id] = hf_pipeline(
            "automatic-speech-recognition",
            model=model_id,
            device=0 if torch.cuda.is_available() else -1,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            chunk_length_s=30,
        )
    pipe = _loaded_pipes[model_id]

    records = []
    t0 = time.time()

    for i, item in enumerate(dataset):
        audio   = item["audio"]
        ref_raw = item.get(TEXT_KEY, "").strip()

        segments = apply_vad(audio["array"], vad_cfg)

        t_inf = time.time()
        segment_texts = []
        for seg in segments:
            if len(seg) < 400:   # ignore for recording less than 25 seconds
                continue
            pipe_kwargs = {}
            if model_type == "whisper" and run_cfg["generate_kwargs"]:
                pipe_kwargs["generate_kwargs"] = run_cfg["generate_kwargs"]
            try:
                out = pipe(
                    {"array": seg, "sampling_rate": audio["sampling_rate"]},
                    **pipe_kwargs,
                )
                segment_texts.append(out["text"].strip())
            except Exception:
                pass
        inf_time = time.time() - t_inf

        hyp_raw  = " ".join(segment_texts).strip()
        ref_norm = normalize_for_wer(ref_raw)
        hyp_norm = normalize_for_wer(hyp_raw)

        w = wer(ref_norm, hyp_norm) if ref_norm else 0.0
        c = cer(ref_norm, hyp_norm) if ref_norm else 0.0

        records.append({
            "id"            : item.get("id", f"sample_{i:04d}"),
            "reference_raw" : ref_raw,
            "reference"     : ref_norm,
            "hypothesis_raw": hyp_raw,
            "hypothesis"    : hyp_norm,
            "wer"           : round(w, 4),
            "cer"           : round(c, 4),
            "duration_s"    : round(len(audio["array"]) / audio["sampling_rate"], 2),
            "inference_s"   : round(inf_time, 3),
            "vad_enabled"   : vad_cfg.get("enabled", False),
            "segments"      : len(segments),
        })

        if (i + 1) % 50 == 0:
            avg_wer = np.mean([r["wer"] for r in records])
            print(f"  [{run_cfg['run_id']}] {i+1}/{len(dataset)}  WER={avg_wer:.2%}  {time.time()-t0:.0f}s")

    return records


for run in RUNS:
    out_path = OUTPUT_DIR / "predictions" / f"predictions_{run['run_id']}.json"

    if out_path.exists():
        print(f"[{run['run_id']}] found on disk, loading...")
        with open(out_path, encoding="utf-8") as f:
            all_records[run["run_id"]] = json.load(f)
        continue

    print(f"\n{'='*55}")
    print(f"[{run['run_id']}] {run['name']}")
    print(f"{'='*55}")

    records = run_inference(run, dataset)
    all_records[run["run_id"]] = records

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    # آزادسازی مدل CTC بعد از هر run (Whisper cache نگه می‌دارد)
    if run.get("model_type") == "ctc":
        del _loaded_pipes[run["model_id"]]
        torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()

[run1] found on disk, loading...
[run2] found on disk, loading...
[run3] found on disk, loading...
[run4] found on disk, loading...
[run5] found on disk, loading...
[run6] found on disk, loading...


## Re-normalize
run this cell and the following when you update the normalizer

In [31]:
print("Re-applying normalize_for_wer to all cached predictions...")

for run in RUNS:
    records = all_records[run["run_id"]]
    n_updated = 0

    for r in records:
        ref_new = normalize_for_wer(r["reference_raw"])
        hyp_new = normalize_for_wer(r["hypothesis_raw"])

        if ref_new != r["reference"] or hyp_new != r["hypothesis"]:
            n_updated += 1

        r["reference"] = ref_new
        r["hypothesis"] = hyp_new
        r["wer"] = round(wer(ref_new, hyp_new) if ref_new else 0.0, 4)
        r["cer"] = round(cer(ref_new, hyp_new) if ref_new else 0.0, 4)

    # overwrite saved JSON so next load is also up to date
    out_path = OUTPUT_DIR / "predictions" / f"predictions_{run['run_id']}.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(records, f, ensure_ascii=False, indent=2)

    print(f"  [{run['run_id']}]  {len(records)} samples  |  {n_updated} re-normalized")

print("\nDone — all_records updated in memory and on disk.")

Re-applying normalize_for_wer to all cached predictions...
  [run1]  871 samples  |  0 re-normalized
  [run2]  871 samples  |  0 re-normalized
  [run3]  871 samples  |  0 re-normalized
  [run4]  871 samples  |  0 re-normalized
  [run5]  871 samples  |  0 re-normalized
  [run6]  871 samples  |  0 re-normalized

Done — all_records updated in memory and on disk.


## Benchmark

In [24]:
def compute_sdi(records):
    S = D = I = H = N = 0
    for r in records:
        try:
            out = process_words(r["reference"], r["hypothesis"])
            S += out.substitutions
            D += out.deletions
            I += out.insertions
            H += out.hits
            N += len(r["reference"].split())
        except Exception:
            pass
    n = max(N, 1)
    refs  = [r["reference"]  for r in records]
    hyps  = [r["hypothesis"] for r in records]
    return {
        "WER"    : round((S+D+I)/n, 4),
        "CER"    : round(cer(refs, hyps), 4),
        "S"      : S, "D": D, "I": I, "N": N,
        "S_rate" : round(S/n, 4),
        "D_rate" : round(D/n, 4),
        "I_rate" : round(I/n, 4),
    }


summary = []
print(f"{'Config':<48} {'WER':>6} {'CER':>6} {'S':>5} {'D':>5} {'I':>5}")
print("-" * 70)

for run in RUNS:
    records   = all_records[run["run_id"]]
    sdi       = compute_sdi(records)
    wer_list  = [r["wer"] for r in records]
    perfect   = sum(1 for w in wer_list if w == 0)

    row = {
        "run_id"                  : run["run_id"],
        "name"                    : run["name"],
        "model_id"                : run["model_id"],
        "beam_size"               : run["generate_kwargs"].get("num_beams", "N/A"),
        "samples"                 : len(records),
        **sdi,
        "wer_mean"                : round(float(np.mean(wer_list)), 4),
        "wer_median"              : round(float(np.median(wer_list)), 4),
        "wer_std"                 : round(float(np.std(wer_list)), 4),
        "perfect_transcriptions"  : perfect,
    }
    summary.append(row)
    print(f"{run['name']:<48} {sdi['WER']:>6.2%} {sdi['CER']:>6.2%}"
          f" {sdi['S']:>5} {sdi['D']:>5} {sdi['I']:>5}")

bench_path = OUTPUT_DIR / "benchmark_results.json"
with open(bench_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

Config                                              WER    CER     S     D     I
----------------------------------------------------------------------
jacktol-whisper-large-v3-ATC | beam=5             6.96%  3.76%   477   189    86
jacktol-whisper-large-v3-ATC | beam=1 (greedy)    7.44%  4.06%   513   204    86
jacktol-whisper-medium.en-ATC | beam=1 (greedy)   8.19%  5.54%   536   201   147
fjmgAI-whisper-large-v3-ATC | beam=1             37.60% 24.09%  2144   935   982
XLS-R-300M | UWB-ATCC+ATCOSIM (CTC)              45.62% 23.18%  3443  1224   260
jacktol-whisper-large-v3-ATC | beam=5 + VAD      12.66%  9.24%   648   618   101


## Normalization

In [25]:
records_r1 = all_records["run1"]

refs_raw  = [r["reference_raw"].lower()  for r in records_r1]
hyps_raw  = [r["hypothesis_raw"].lower() for r in records_r1]
wer_raw   = wer(refs_raw, hyps_raw)

refs_norm = [r["reference"]  for r in records_r1]
hyps_norm = [r["hypothesis"] for r in records_r1]
wer_norm  = wer(refs_norm, hyps_norm)

improvement = wer_raw - wer_norm
print(f"WER raw : {wer_raw:.2%}")
print(f"WER norm: {wer_norm:.2%}")
print(f"delta   : {improvement:+.2%}")

examples = [
    "descend flight level one eight zero",
    "squawk four five six seven",
    "contact one two seven decimal four",
    "runway two eight left",
    "climb to flight level three five zero",
    "niner thousand feet",
    "heading two four zero",
]

norm_report = {
    "wer_without_normalization": round(wer_raw, 4),
    "wer_with_normalization"   : round(wer_norm, 4),
    "improvement"              : round(improvement, 4),
    "examples_before_after"    : [
        {"before": ex, "after_wer": normalize_for_wer(ex), "after_display": normalize_for_display(ex)}
        for ex in examples
    ],
}
norm_path = OUTPUT_DIR / "normalization_report.json"
with open(norm_path, "w", encoding="utf-8") as f:
    json.dump(norm_report, f, ensure_ascii=False, indent=2)

WER raw : 8.98%
WER norm: 6.96%
delta   : +2.01%


## Error Analysis

In [26]:
CALLSIGN_AIRLINES = {
    "lufthansa","swiss","iberia","ryanair","easyjet","delta","united",
    "american","british","france","alitalia","klm","turkish","austrian",
    "finnair","brussels","tap","lot","aegean","croatia",
}
NUMBER_WORDS = {
    "zero","one","two","three","four","five","six","seven","eight","nine",
    "ten","eleven","twelve","thirteen","fourteen","fifteen","sixteen",
    "seventeen","eighteen","nineteen","twenty","thirty","forty","fifty",
    "sixty","seventy","eighty","ninety","hundred",
}
RUNWAY_WORDS = {"runway","approach","ils","localizer","glide"}
COMMAND_WORDS = {
    "descend","climb","maintain","heading","turn","contact","cleared",
    "squawk","frequency","report","expedite","hold","direct","ident",
}

SUGGESTIONS = {
    "callsign"     : "hot-word boosting or callsign lexicon fine-tune",
    "flight_level" : "check text normalization for alternate number forms",
    "runway"       : "runway tokens usually paired with digits — verify normalization",
    "command"      : "fixed ATC vocabulary — domain LM may help",
    "number"       : "digits are critical in ATC — consistent normalization required",
    "other"        : "check audio quality — noise or accent may be the issue",
}

def categorize(word: str) -> str:
    w = word.lower()
    if any(a in w for a in CALLSIGN_AIRLINES): return "callsign"
    if w in RUNWAY_WORDS:  return "runway"
    if w in COMMAND_WORDS: return "command"
    if w in NUMBER_WORDS:  return "number"
    if w in {"flight","level"}: return "flight_level"
    return "other"

def severity(cat: str) -> str:
    if cat in {"callsign","runway","flight_level"}: return "critical"
    if cat in {"command","number"}:                 return "moderate"
    return "minor"


errors = []
for rec in sorted(records_r1, key=lambda r: r["wer"], reverse=True):
    if len(errors) >= 35: break
    ref_w = rec["reference"].split()
    hyp_w = rec["hypothesis"].split()
    try:
        out = process_words(rec["reference"], rec["hypothesis"])
    except Exception:
        continue
    for al in out.alignments[0]:
        if len(errors) >= 35: break
        if al.type == "substitute":
            rw = ref_w[al.ref_start_idx] if al.ref_start_idx < len(ref_w) else ""
            hw = hyp_w[al.hyp_start_idx] if al.hyp_start_idx < len(hyp_w) else ""
            cat = categorize(rw)
            errors.append({
                "sample_id"   : len(errors)+1,
                "file_id"     : rec["id"],
                "error_type"  : "S",
                "category"    : cat,
                "severity"    : severity(cat),
                "ref_word"    : rw,
                "hyp_word"    : hw,
                "ref_sentence": rec["reference"],
                "hyp_sentence": rec["hypothesis"],
                "suggestion"  : SUGGESTIONS[cat],
            })
        elif al.type == "delete":
            rw = ref_w[al.ref_start_idx] if al.ref_start_idx < len(ref_w) else ""
            cat = categorize(rw)
            errors.append({
                "sample_id"   : len(errors)+1,
                "file_id"     : rec["id"],
                "error_type"  : "D",
                "category"    : cat,
                "severity"    : severity(cat),
                "ref_word"    : rw,
                "hyp_word"    : "[deleted]",
                "ref_sentence": rec["reference"],
                "hyp_sentence": rec["hypothesis"],
                "suggestion"  : SUGGESTIONS[cat],
            })

cat_counts = Counter(e["category"]  for e in errors)
sev_counts = Counter(e["severity"]  for e in errors)
typ_counts = Counter(e["error_type"] for e in errors)

print(f"samples: {len(errors)}")
print(f"types  : {dict(typ_counts)}")
print(f"cats   : {dict(cat_counts)}")
print(f"sev    : {dict(sev_counts)}")

samples: 35
types  : {'S': 25, 'D': 10}
cats   : {'other': 30, 'command': 2, 'number': 3}
sev    : {'minor': 30, 'moderate': 5}


## Save

In [27]:
ea_json = OUTPUT_DIR / "error_analysis.json"
with open(ea_json, "w", encoding="utf-8") as f:
    json.dump(errors, f, ensure_ascii=False, indent=2)

ea_txt = OUTPUT_DIR / "error_analysis.txt"
with open(ea_txt, "w", encoding="utf-8") as f:
    f.write("ATC-ASR Error Analysis\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"samples: {len(errors)}\n")
    f.write(f"categories: {dict(cat_counts)}\n")
    f.write(f"severity: {dict(sev_counts)}\n\n")
    f.write("=" * 60 + "\n\n")
    for e in errors:
        f.write(f"[{e['error_type']}] #{e['sample_id']} | {e['category']} | {e['severity']}\n")
        f.write(f"  REF: {e['ref_sentence']}\n")
        f.write(f"  HYP: {e['hyp_sentence']}\n")
        f.write(f"  error: '{e['ref_word']}' -> '{e['hyp_word']}'\n")
        f.write(f"  note: {e['suggestion']}\n\n")

log_path = OUTPUT_DIR / "execution_log.json"
log = {
    "timestamp"   : time.strftime("%Y-%m-%d %H:%M:%S"),
    "dataset"     : DATASET_ID,
    "samples"     : len(dataset),
    "runs"        : [
        {
            "model"     : r["model_id"],
            "run_id"    : r["run_id"],
            "name"      : r["name"],
            "beam_size" : run["generate_kwargs"].get("num_beams", "N/A"),
            "wer"       : next(s["WER"] for s in summary if s["run_id"] == r["run_id"]),
            "cer"       : next(s["CER"] for s in summary if s["run_id"] == r["run_id"]),
        }
        for r in RUNS
    ],
}
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(log, f, ensure_ascii=False, indent=2)